## Building Web Scraper with Agents

#### Step 1: Web scraper needs search Engine

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

result = search.run("What is a transformer model in deep learning?")
print(result)


In deep learning , the transformer is an artificial neural network architecture based on the multi-head attention mechanism, in which text is converted to numerical representations called tokens, and each token is converted into a vector via lookup from a word embedding table. [1] A transformer model is a type of deep learning model that has quickly become fundamental in natural language processing (NLP) and other machine learning (ML) tasks. Oct 18, 2025 · Transformers are a type of deep learning model that utilizes self-attention mechanisms to process and generate sequences of data efficiently . They capture long-range dependencies and contextual relationships making them highly effective for tasks like language modeling, machine translation and text generation. Feb 4, 2025 · What is a Transformer ? The transformer model , introduced in the paper Attention Is All You Need by Vaswani et al. (2017), marked a significant departure from previous sequence-to-sequence... Transformers are p

In [ ]:
import os
from langchain_groq import ChatGroq
from langchain_core.tools import Tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
import uuid

# ✅ Set your Groq API key
os.environ["GROQ_API_KEY"] = ""  # replace safely

# ✅ LLM with Groq (tool-call capable)
llm = ChatGroq(
    model="llama-3.3-70b-versatile",  # supports tool calling
    temperature=0.1
)

# ✅ Define tools
duckduckgo = DuckDuckGoSearchRun()
wikipedia = WikipediaAPIWrapper()

tools = [
    Tool(
        name="duckduckgo_search",
        func=lambda query: duckduckgo.run({"query": query}),
        description="Search the web for current facts using DuckDuckGo"
    ),
    Tool(
        name="wikipedia_search",
        func=lambda query: wikipedia.run(query),
        description="Get factual information from Wikipedia"
    )
]

# ✅ Create LangGraph ReAct agent
agent = create_react_agent(model=llm, tools=tools)

# ✅ Run the query
query = "whats today's date"

response = agent.invoke({
    "messages": [HumanMessage(content=query)]
})

print(response["messages"][-1].content)


C:\Users\dhanu\AppData\Local\Temp\ipykernel_29056\1245253307.py:37: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(model=llm, tools=tools)


Today's date is January 27, 2026.


🔁 Step-by-Step: What Happens Internally
🧠 1. You Send a Query

Example:

query = "Who is the President of Italy in 2024? Who was the previous one?"
agent.invoke({"messages": [HumanMessage(content=query)]})

🧠 2. Groq's LLM Receives the Prompt

The agent uses a ReAct prompt structure (Reasoning + Acting) like this:

Thought: Do I know the answer? If not, I should use a tool.
Action: duckduckgo_search
Action Input: current president of Italy


The LLM decides what to do based on:

Keywords in the query

Tool descriptions you provided

ReAct prompt logic (structured to "think" → "act" → "observe" → "repeat")

🔧 3. Tool Calling Happens Automatically

Let’s say the LLM chooses:

Action: duckduckgo_search
Action Input: Who is the President of Italy in 2024?


LangChain takes that and calls:

duckduckgo.run({"query": "Who is the President of Italy in 2024?"})


The tool returns something like:

“The President of Italy as of 2024 is Sergio Mattarella...”

LangChain inserts this into the reasoning flow as:

Observation: The President of Italy as of 2024 is Sergio Mattarella...


Then the LLM continues processing with this new information.

🔁 4. (Optional) Multiple Tools Might Be Used

If your query has multiple parts like:

"Who is the President of Italy in 2024? Who was the previous one?"

The LLM might:

✅ Use DuckDuckGo to get the current president

✅ Use Wikipedia to find the list of past presidents

Tool choice is based on:

Tool descriptions you provided

Clarity and structure of the question

Internal confidence and relevance checks by the LLM

🧠 5. Final Answer Generation

Once all tool results are in, the LLM assembles a final message like:

The current President of Italy in 2024 is Sergio Mattarella.
The previous president was Giorgio Napolitano.

You can access this via:

response["messages"][-1].content

🧭 How It Decides Which Tool to Use

The LLM evaluates:

Relevance of the tool description to the query

Whether the answer is already known (pretrained knowledge)

If the info might be recent or dynamic → prefers DuckDuckGo

If the info is encyclopedic/static → prefers Wikipedia

🧠 Example LLM Thought Flow
Hmm, this is about recent political events → Use duckduckgo_search
Now I need historical data → Use wikipedia_search

🔍 Observe the Agent's Reasoning

Enable verbose logging to inspect step-by-step reasoning:

agent = create_react_agent(model=llm, tools=tools, verbose=True)


You’ll see output like:

Thought: I need to find out who the president is.
Action: duckduckgo_search
Input: Who is the President of Italy in 2024?

Observation: Sergio Mattarella is the current President...

Thought: Now let me find the previous one.
Action: wikipedia_search
Input: List of Presidents of Italy

Observation: Giorgio Napolitano was before him.

Final Answer: ...

🔧 Summary
Component	Role
LLM (Groq)	Thinks, chooses tools, generates final answer
LangChain Agent	Manages reasoning loop (Thought → Action → Observation)
DuckDuckGo Tool	Real-time web search
Wikipedia Tool	Retrieves factual knowledge
Your Tool Descriptions	Guide the LLM on which tool to use and when
✅ Want to Extend It?

Add custom tools (PDF reader, SQL query, CSV summarizer, etc.)

Stream responses or build a chatbot

Use it with voice or frontend integrations

Let me know and I’ll help you set it up!